In [6]:
import tensorflow as tf
from pathlib import Path

print("TF:", tf.__version__)          # note this — goes into requirements.txt later
print("GPU:", tf.config.list_physical_devices("GPU"))

# find repo root regardless of where the kernel starts
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR  = ROOT / "Downloads" / "ai-deepfakedetection-main" / "backend" / "train" / "dataset"         # dataset
MODEL_DIR = ROOT / "Downloads" / "ai-deepfakedetection-main" / "backend" /"saved_models"  # output
print("Repo root:", ROOT)

TF: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Repo root: /Users/priyankareddy


In [7]:
import zipfile, os

if not (DATA_DIR / "train").exists():
    !kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
    with zipfile.ZipFile("cifake-real-and-ai-generated-synthetic-images.zip") as z:
        z.extractall(DATA_DIR)

print(os.listdir(DATA_DIR))    # expect: ['train', 'test']

['.DS_Store', 'test', 'train']


In [9]:
IMG_SIZE = 32
BATCH = 256

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / "train"), validation_split=0.1, subset="training", seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="binary")

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / "train"), validation_split=0.1, subset="validation", seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH, label_mode="binary")

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    str(DATA_DIR / "test"), image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
    label_mode="binary", shuffle=False)

train_ds = train_ds_raw.cache().prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds_raw.cache().prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds_raw.prefetch(tf.data.AUTOTUNE)

print(train_ds_raw.class_names)   # ['FAKE', 'REAL'] → 0 = FAKE, 1 = REAL

Found 100000 files belonging to 2 classes.
Using 90000 files for training.


2026-09-19 15:49:38.996732: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-09-19 15:49:38.996766: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-09-19 15:49:38.996774: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-09-19 15:49:38.996803: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-09-19 15:49:38.996813: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 100000 files belonging to 2 classes.
Using 10000 files for validation.
Found 20000 files belonging to 2 classes.
['FAKE', 'REAL']


In [10]:
from tensorflow.keras import layers

base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

data_aug = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
])

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = data_aug(inputs)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)   # P(REAL)

model = tf.keras.Model(inputs, outputs)
model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 1, 1, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

history1 = model.fit(train_ds, validation_data=val_ds, epochs=5)

Epoch 1/5


/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-09-19 15:49:56.536484: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


352/352 ━━━━━━━━━━━━━━━━━━━━ 43s 100ms/step - accuracy: 0.7237 - auc: 0.7987 - loss: 0.5524 - val_accuracy: 0.7988 - val_auc: 0.8818 - val_loss: 0.4668
Epoch 2/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 30s 86ms/step - accuracy: 0.7598 - auc: 0.8387 - loss: 0.4995 - val_accuracy: 0.8151 - val_auc: 0.8954 - val_loss: 0.4377
Epoch 3/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 30s 86ms/step - accuracy: 0.7669 - auc: 0.8471 - loss: 0.4866 - val_accuracy: 0.8239 - val_auc: 0.9009 - val_loss: 0.4239
Epoch 4/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 27s 78ms/step - accuracy: 0.7690 - auc: 0.8493 - loss: 0.4830 - val_accuracy: 0.8285 - val_auc: 0.9046 - val_loss: 0.4174
Epoch 5/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 30s 87ms/step - accuracy: 0.7717 - auc: 0.8515 - loss: 0.4798 - val_accuracy: 0.8266 - val_auc: 0.9060 - val_loss: 0.4151


In [12]:
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

history2 = model.fit(
    train_ds, validation_data=val_ds, epochs=5,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True)])

Epoch 1/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 49s 122ms/step - accuracy: 0.7867 - auc: 0.8687 - loss: 0.4532 - val_accuracy: 0.8448 - val_auc: 0.9231 - val_loss: 0.3638
Epoch 2/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 43s 122ms/step - accuracy: 0.8055 - auc: 0.8862 - loss: 0.4252 - val_accuracy: 0.8513 - val_auc: 0.9300 - val_loss: 0.3475
Epoch 3/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 41s 117ms/step - accuracy: 0.8138 - auc: 0.8950 - loss: 0.4096 - val_accuracy: 0.8563 - val_auc: 0.9345 - val_loss: 0.3339
Epoch 4/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 39s 112ms/step - accuracy: 0.8198 - auc: 0.9003 - loss: 0.3996 - val_accuracy: 0.8595 - val_auc: 0.9375 - val_loss: 0.3293
Epoch 5/5
352/352 ━━━━━━━━━━━━━━━━━━━━ 43s 121ms/step - accuracy: 0.8241 - auc: 0.9047 - loss: 0.3910 - val_accuracy: 0.8641 - val_auc: 0.9398 - val_loss: 0.3237


In [15]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

loss, acc, auc = model.evaluate(test_ds)
print(f"Test accuracy: {acc:.4f} | AUC: {auc:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds]).ravel()
y_pred = (model.predict(test_ds).ravel() > 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=["FAKE", "REAL"]))
print(confusion_matrix(y_true, y_pred))

79/79 ━━━━━━━━━━━━━━━━━━━━ 7s 82ms/step - accuracy: 0.8561 - auc: 0.9360 - loss: 0.3340
Test accuracy: 0.8561 | AUC: 0.9360


2026-09-19 16:00:36.336986: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


79/79 ━━━━━━━━━━━━━━━━━━━━ 9s 94ms/step
              precision    recall  f1-score   support

        FAKE       0.83      0.90      0.86     10000
        REAL       0.89      0.81      0.85     10000

    accuracy                           0.86     20000
   macro avg       0.86      0.86      0.86     20000
weighted avg       0.86      0.86      0.86     20000

[[8997 1003]
 [1876 8124]]


In [16]:
import json

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_DIR / "model.keras")

meta = {
    "model": "EfficientNetB0",
    "input_size": [IMG_SIZE, IMG_SIZE],
    "labels": {"0": "FAKE", "1": "REAL"},
    "preprocessing": "RGB, resize to 32x32, keep values 0-255 (no rescaling)",
    "output": "sigmoid, P(REAL)",
    "test_accuracy": float(acc),
}
with open(MODEL_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", list(MODEL_DIR.iterdir()))

Saved: [PosixPath('/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/saved_models/.gitkeep'), PosixPath('/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/saved_models/model.keras'), PosixPath('/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/saved_models/meta.json')]
